# ಪಾಠ 13 - ಏಜೆಂಟ್ ಸ್ಮರಣೆ


## ಸೆಟ್‌ಅಪ್

ಈ ನೋಟ್ಬುಕ್ **ಪರ್ಸಿಸ್ಟೆಂಟ್ ಮೆಮೊರಿ** ಬಳಸಿ **Microsoft Agent Framework** (MAF) తో ಸಹಯಾತ್ರಿ ಬುಕಿಂಗ್ ಏಜೆಂಟ್ ಅನ್ನು ಹೇಗೆ ನಿರ್ಮಿಸಬಹುದು ಎಂದು ತೋರಿಸುತ್ತದೆ.

ನೀವು ಏಜೆಂಟ್ ಮೆಮೊರಿಯ ವಿಭಿನ್ನ ತರಗತಿಗಳು — ವರ್ಕಿಂಗ್, ಶಾರ್ಟ್-ಟರ್ಮ್ ಮತ್ತು ಲಾಂಗ್-ಟರ್ಮ್ — ಸಂಭಾಷಣೆಗಳಲ್ಲಿನ ಮಾಹಿತಿಯನ್ನು ಏಜೆಂಟ್ ఎలా ಸಂಗ್ರಹಿಸಿ ಬಳಸುತ್ತದೆಯೋ ಆ ಕುರಿತು ತಿಳಿಯಲಿದ್ದೀರಿ.

**ಹಾಜರಾತು ಶರತ್ತುಗಳು:**
- ಚಾಟ್ನ ಮಾದರಿಯನ್ನು ನಿಯೋಜಿಸಿರುವ ಮೈಕ್ರೋಸಾಫ್ಟ್ ಫೌಂಡ್ರಿ ಪ್ರಾಜೆಕ್ಟ್ (ಉದಾ: `gpt-5-mini`).
- ಅಜುರ್ CLI ನಲ್ಲಿ ಲಾಗ್ ಇನ್ ಆಗಿದ್ದಿರಿ — ನಿಮ್ಮ ಟರ್ಮಿನಲ್‌ನಲ್ಲಿ `az login` ರನ್ ಮಾಡಿ.
- `AZURE_AI_PROJECT_ENDPOINT` — ನಿಮ್ಮ ಮೈಕ್ರೋಸಾಫ್ಟ್ ಫೌಂಡ್ರಿ ಪ್ರಾಜೆಕ್ಟ್ ಎಂಡ್ಪಾಯಿಂಟ್.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ನಿಯೋಜಿಸಿದ ನಿಮ್ಮ ಮಾದರಿಯ ಹೆಸರು.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ಏಜೆಂಟ್ ಸ್ಮृति ಪ್ರಕಾರಗಳು

AI ಏಜೆಂಟ್‌ಗಳು ವಿಭಿನ್ನ ರೀತಿಯ ಸ್ಮೃತಿಗಳನ್ನು ಬಳಸಿಕೊಳ್ಳಬಹುದು, ಪ್ರತಿಯೊಂದು ವಿಭಿನ್ನ ಉದ್ದೇಶವನ್ನು ಪೂರೈಸುತ್ತದೆ:

### ಕಾರ್ಯಾಚರಣೆಯ ಸ್ಮೃತಿ
ಸಂಭಾಷಣಾ ಧಾರೆಸ್ವರೂಪ — ಒಂದೇ ಸೆಷನ್‌ನಲ್ಲಿ ವಿನಿಮಯವಾಯ್ತು ಸಂದೇಶಗಳು. ಏಜೆಂಟ್ ಅದೇ ಧಾರೆಗಳಲ್ಲಿ ಹಿಂದಿನ ಸಂದೇಶಗಳನ್ನು ಉಲ್ಲೇಖಿಸಿ ಸुसಂಬಂಧವನ್ನು ಕಾಪಾಡಿಕೊಳ್ಳುತ್ತದೆ. MAF ನಲ್ಲಿ ಇದು **`agent.create_session()`** ಮೂಲಕ ಸೃಷ್ಟಿಸಲಾಗುತ್ತದೆ, ಇದು `AgentSession` ಅನ್ನು ಹಿಂತಿರುಗಿಸುತ್ತದೆ.

### ಸಣ್ಣ ಅವಧಿಯ ಸ್ಮೃತಿ
ಕಾರ್ಯ ಅಥವಾ ಸೆಷನ್ ಅವಧಿಗೆ ತಡೆಯುವ ಮಾಹಿತಿ ಆದರೆ ಶಾಶ್ವತವಾಗಿ ಉಳಿಸಿಕೊಳ್ಳಲಾಗುವುದಿಲ್ಲ. ಉದಾಹರಣೆಗೆ, ಏಜೆಂಟ್ ಬಹು-ತಿರುವು ಯೋಜನೆ ಸಂಭಾಷಣೆಯ ವೇಳೆ ವಾಸ್ತವಾಂಶಗಳನ್ನು ಸಂಗ್ರಹಿಸಿ ಅಂತಿಮ ಪ್ರಯಾಣ ಯೋಜನೆ ಸೃಷ್ಟಿಸಲು ಬಳಸದಬಹುದು.

### ದೀರ್ಘಾವಧಿಯ ಸ್ಮೃತಿ
ಸೆಷನ್‌ಗಳ **ಮಧ್ಯೆ** ಉಳಿಯುವ ಪ್ರವೃತ್ತಿಗಳು ಮತ್ತು ವಾಸ್ತವಾಂಶಗಳು. ಮರಳುವ ಬಳಕೆದಾರರು ತಮ್ಮ ಆಹಾರ ನಿರ್ಬಂಧಗಳು ಅಥವಾ ಪ್ರವಾಸ ಶೈಲಿಯನ್ನು ಮರುಕಳಿಸಬಾರದು. ದೀರ್ಘಾವಧಿಯ ಸ್ಮೃತಿಗೆ ಸಾಮಾನ್ಯವಾಗಿ ಹೊರಗಿನ ಸಂಗ್ರಹಣಾ ಪರಿಕರಗಳಿವೆ — ಡೇಟಾಬೇಸ್, ಫೈಲ್ ಅಥವಾ ವೆಕ್ಟರ್ ಸೂಚಿಮಾಡಿಕೆ — ಮತ್ತು ಅವುಗಳನ್ನು ಏಜೆಂಟ್ ಗೆ ಸಾಧನಗಳ ಮೂಲಕ ತೋರಿಸಲಾಗುತ್ತದೆ.


## ಸೆಷನ್‌ಗಳೊಂದಿಗೆ ಕಾರ್ಯನಿರ್ವಹಿಸುವ ಸ್ಮರಣೆ

ಸ್ಮರಣೆಯ ಸರಳ ಸ್ವರೂಪವು ಸಂಭಾಷಣೆ ಸೆಷನ್ ಆಗಿದೆ. ನೀವು ಅದೇ ಸೆಷನ್ (`agent.create_session()` ಮೂಲಕ ರಚಿಸಲ್ಪಟ್ಟಿದೆ) ಅನ್ನು ಸರಳವಾಗಿ ಮುಂದುವರಿದ `agent.run()` ಕರೆಗಳಿಗೆ ನೀಡಿದಾಗ, ಏಜೆಂಟ್ ಆ ಸಂಭಾಷಣೆಯ ಸಂಪೂರ್ಣ ಇತಿಹಾಸವನ್ನು ನೋಡಿಕೊಳ್ಳುತ್ತೆ ಮತ್ತು ಹಿಂದಿನ ವಿವರಗಳನ್ನು ನೆನಪಿನಲ್ಲಿಡಬಹುದು.

ಬನ್ನಿ, ಪ್ರವಾಸ ಏಜೆಂಟ್ನನ್ನು ರಚಿಸುವುದಾಗಿ ಮತ್ತು ಕಾರ್ಯನಿರ್ವಹಿಸುವ ಸ್ಮರಣೆ ಅನ್ನು ಪ್ರದರ್ಶಿಸುವುದಾಗಿ ಮಾಡೋಣ.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ಏಜೆಂಟ್ ಬಜೆಟ್‌ನನ್ನು ಸರಿಯಾಗಿ ನೆನಪಿಗೆ ತಂದಿದ್ದಾನೆ ಏಕೆಂದರೆ ಎರಡೂ ಸಂದೇಶಗಳು ಒಂದೇ ಸೆಷನ್ ಅನ್ನು ಹಂಚಿಕೊಂಡಿವೆ. ಇದನ್ನು **ಕೆಲಸದ ನೆನಪು** ಎಂದು ಕರೆಯುತ್ತಾರೆ — ಇದು ಸೆಷನ್‌ನ ಆಯುಷ್ಯಾವಧಿಗಾಗಿ ಮಾತ್ರ ಉಂಟಾಗುತ್ತದೆ.

### ಹೊಸ ತಂತಿ (ಥ್ರೆಡ್) ನಲ್ಲಿ ಏನು ಸಂಭವಿಸುತ್ತದೆ?

ನಾವು **ಹೊಸ** ಸೆಷನ್ ಸೃಷ್ಟಿಸಿದರೆ, ಏಜೆಂಟ್ ಹಿಂದಿನ ಸಂವಾದವನ್ನು ನೆನಪಿಸಿಕೊಂಡಿರುತ್ತಾನೆ:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## ದೀರ್ಘಾವಧಿ ಸ್ಮರಣೆ ಮಾದರಿ

ಬಳಕೆದಾರರ ಮೆಚ್ಚುಗೆಗಳನ್ನು **ಅವಧಿಗಳ ಹಿಂದೆ** ಮರೆತುಬಿಡದಿರುವುದಕ್ಕಾಗಿ, ನಾವು ಸಂಭಾಷಣೆ ಕೂರ್ಮದ ಹೊರಗಿರುವ ಒಂದು ಸ್ಥಿರ ಸಂಗ್ರಹಣೆಯನ್ನು ಬೇಕಾಗುತ್ತದೆ. ಏಜೆಂಟ್ ಈ ಸಂಗ್ರಹಣೆಗೆ **ಸಾಧನಗಳ** ಮೂಲಕ ಪ್ರವೇಶಿಸುತ್ತದೆ — ಮಾಹಿತಿ ಉಳಿಸಲು ಮತ್ತು ತೆಗೆದುಕೊಳ್ಳಲು ಕರೆ ಮಾಡಬಹುದಾದ ಕಾರ್ಯಗಳು.

ಕೆಳಗೆ ನಾವು ಸಾದಾರಣ ಮನಸ್ಸಿನ ಮೆಚ್ಚುಗೆ ಸಂಗ್ರಹಣೆಯನ್ನು ಜಾರಿಗೆ ತರುತ್ತೇವೆ (ಉತ್ಪಾದನೆಯಲ್ಲಿ ನೀವು ಇದನ್ನು ಡೇಟಾಬೇಸ್ ಅಥವಾ ವೆಕ್ಟರ್ ಸೂಚ್ಯಂಕದ ಮೂಲಕ ಬೆಂಬಲಿಸುವಿರಿ) ಮತ್ತು ಇದನ್ನು ಏಜೆಂಟ್ ಬಳಸಬಹುದಾದ ಸಾಧನಗಳಾಗಿ ಬಹಿರಂಗ ಮಾಡುತ್ತೇವೆ.

### ವಾಸ್ತುಶಿಲ್ಪ
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### ಸಂದರ್ಭ 1 — ಮೊಟ್ಟಮೊದಲು ಬಳಕೆದಾರನು ವಾರ್ಷಿಕೋತ್ಸವದ ಪ್ರವಾಸವನ್ನು ಬುಕ್ ಮಾಡುತ್ತಾನೆ

ಸಾರಾಹ್ ಮೊಟ್ಟಮೊದಲಿಗೆ ಭೇಟಿ ನೀಡುತ್ತಾರೆ. ಏಜೆಂಟ್ ಅವರ ಆದ್ಯತೆಗಳನ್ನು ಉಪಕರಣಗಳ ಮೂಲಕ ಸಂಗ್ರಹಿಸಿ ಅವುಗಳನ್ನು ಹೋಟೆல್ಗಳನ್ನು ಶಿಫಾರಸು ಮಾಡಲು ಬಳಸಬೇಕು.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### ದೃಶ್ಯ 2 — ಸಚಾ ಕೆಲವು ವಾರಗಳ ನಂತರ ಮರಳುತ್ತದೆ

ಸಚಾ ಒಂದು **ಹೊಸ ತಂತಿಯನ್ನು** ಶುರುಮಾಡುತ್ತದೆ (ಹೊಸ ಸೆಷನ್ ಅನ್ನು ಅನುಕರಿಸುತ್ತಿದೆ). ಕೆಲಸದ ಸ್ಮರಣೆ ಖಾಲಿರಿರುತ್ತದೆ, ಆದರೆ ದೀರ್ಘಕಾಲಿಕ ಇಚ್ಛಿತ ಸಂಗ್ರಹದಲ್ಲಿ ಇನ್ನೂ ಅವಳ ಮಾಹಿತಿಯಿದೆ. ಏಜೆಂಟ್ ಅದನ್ನು ಪಡೆದು ವೈಯಕ್ತಿಕ ಶಿಫಾರಸುಗಳಿಗಾಗಿ ಬಳಸಿಕೊಳ್ಳಬೇಕು.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Summary

In this lesson you learned three types of agent memory and how to implement them with the Microsoft Agent Framework:

| Memory Type | MAF Mechanism | Lifetime |
|---|---|---|
| **Working** | `agent.create_session()` | Single conversation |
| **Short-term** | Accumulated context within a thread | Single task / session |
| **Long-term** | External store accessed via `@tool` functions | Across sessions |

### Key Takeaways
1. **`agent.create_session()`** provides working memory — the agent sees the full conversation history within a session.
2. **New sessions lose context** — without long-term memory the agent cannot recall past conversations.
3. **`@tool` functions** bridge the gap — they let the agent save and retrieve information from a persistent store.
4. **Personalization improves over time** — the more preferences are stored, the better the agent's recommendations.

### Real-World Applications
- **Customer Service**: Remember customer history and preferences
- **Personal Assistants**: Maintain context across days or weeks
- **Healthcare**: Track patient information and preferences
- **E-commerce**: Personalized shopping based on history

### Next Steps
- Replace the in-memory dict with a database or vector store (e.g. Azure AI Search)
- Add memory expiration for time-sensitive information
- Build multi-agent systems with shared memory
- Explore the Cognee notebook for knowledge-graph-backed memory


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
